In [ ]:
import torch
import torch.nn as nn 
# -------------------------------------------------------------------------------------
# [arXiv:1706.03762] Vaswani et al., "Attention Is All You Need"
# Learnable positional embedding (instead of sinusoidal encoding)
# -------------------------------------------------------------------------------------
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# -------------------------------------------------------------------------------------
# [arXiv:2304.14802] ResiDual: "Residual Connections are Natural Preconditioners"
# Dual LayerNorm (Pre + Post) in attention and FFN sublayers
# Improves gradient flow in deeper Transformers
# -------------------------------------------------------------------------------------
class ResiDualBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.ln1_pre = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ln1_post = nn.LayerNorm(d_model)

        self.ln2_pre = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),  # [arXiv:1606.08415] Gaussian Error Linear Unit
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.ln2_post = nn.LayerNorm(d_model)

    def forward(self, x):
        # Self-attention block
        x_norm = self.ln1_pre(x)
        attn_output, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.ln1_post(attn_output)

        # Feedforward block
        x_norm = self.ln2_pre(x)
        ff_output = self.ffn(x_norm)
        x = x + self.ln2_post(ff_output)
        return x
        
# -------------------------------------------------------------------------------------
# [arXiv:2111.11432] ConvNeXt / [arXiv:2201.03545] MetaFormer
# Residual FFN with GELU, acts as decoder bottleneck block
# -------------------------------------------------------------------------------------
class ResidualFFN(nn.Module):
    def __init__(self, dim, hidden_dim=None, dropout=0.1):
        super().__init__()
        hidden_dim = hidden_dim or dim * 4
        self.norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x):
        return x + self.ffn(self.norm(x))       
        
# -------------------------------------------------------------------------------------
# [arXiv:1706.03762] Transformer + [arXiv:2304.14802] ResiDual
# Time series regression with ResiDual attention blocks and residual decoder
# -------------------------------------------------------------------------------------
class TransformerPredictor(nn.Module):
    def __init__(self, input_size=7, output_size=4, d_model=128, nhead=8, num_layers=7, dim_feedforward=512, dropout=0.1, output_len=1, use_attention_pool=True):
        super().__init__()
        self.output_size = output_size
        self.output_len = output_len
        self.use_attention_pool = use_attention_pool

        # [arXiv:2010.11929] LayerNorm before projection
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_size),
            nn.SiLU(),
            nn.Linear(input_size, d_model//4),
            
            nn.LayerNorm(d_model//4),
            nn.SiLU(),
            nn.Linear(d_model//4, d_model//2),

            nn.LayerNorm(d_model//2),
            nn.SiLU(),
            nn.Linear(d_model//2, d_model)
        )
        self.pos_encoder = LearnablePositionalEncoding(d_model)

        # [arXiv:2304.14802] ResiDual encoder blocks
        self.encoder_layers = nn.ModuleList([
            ResiDualBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

        # [arXiv:2005.12872] Attention-based pooling layer (like Set Transformer)
        if self.use_attention_pool:
            self.attn_pool = nn.Sequential(
                nn.Linear(d_model, d_model//2),
                nn.Tanh(),
                nn.Linear(d_model//2, 1)
            )

        # Decoder MLP with residuals
        self.decoder_input = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 256),
            nn.SiLU()  # [arXiv:1606.08415] Swish-1 variant
        )
        # Residual FFN decoder blocks
        self.res_blocks = nn.Sequential(
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
        )
        # Final projection to output space
        self.decoder_output = nn.Sequential(
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Linear(256, 128),

            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Linear(128, 64),

            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Linear(64, output_size * output_len)
        )

    def forward(self, x):
        # Encode
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        for layer in self.encoder_layers:
            x = layer(x)

        # Pooling
        if self.use_attention_pool:
            attn_scores = torch.softmax(self.attn_pool(x), dim=1)  # (B, S, 1)
            x_pooled = (attn_scores * x).sum(dim=1)  # (B, d_model)
        else:
            x_pooled = x.mean(dim=1)

        # Decode
        x_pooled = self.decoder_input(x_pooled)
        x_pooled = self.res_blocks(x_pooled)
        out = self.decoder_output(x_pooled)

        # Reshape
        return out.view(-1, self.output_len, self.output_size)